# CityLens · Sorocaba — Pipeline no Google Colab

**Como usar (não precisa instalar nada no seu PC):**
1. No menu de cima: **Ambiente de execução → Executar tudo** (ou `Ctrl+F9`).
2. Espere ~2–4 min. No fim aparece o **mapa das 6 zonas** e uma tabela.

Este notebook baixa a malha de setores censitários do **IBGE** (via `geobr`),
descobre a posição de cada bairro pelo **OpenStreetMap** (limitado aos limites de
Sorocaba) e atribui cada setor à **zona do bairro mais próximo**. As 6 zonas são
as oficiais do zoneamento da Prefeitura/Urbes. Regra do projeto: **não inventar
dado** — bairro que não for encontrado (ou que cair fora de Sorocaba) é avisado e
ignorado, nunca chutado.


In [ ]:
# 1) Instala as bibliotecas (só nesta sessão do Colab)
!pip -q install geobr geopandas matplotlib requests 2>/dev/null
print("Bibliotecas prontas.")

In [ ]:
# 2) As 6 zonas oficiais e seus bairros (transcritos do mapa oficial)
ZONAS = {
    "centro": {
        "nome": "Centro",
        "bairros": [
            "Centro"
        ]
    },
    "norte": {
        "nome": "Zona Norte",
        "bairros": [
            "Jardim Botucatu",
            "Jardim Maria Eugênia",
            "Parque Laranjeiras",
            "Jardim Guaíba",
            "Jardim São Guilherme",
            "Jardim Santa Marina",
            "Parque São Bento",
            "Jardim Novo Horizonte",
            "Jardim Maria Antônia Prado",
            "Jardim Guadalupe",
            "Vila Nova Sorocaba",
            "Vila Angélica",
            "Jardim Húngaros",
            "Jardim Brasilândia",
            "Vila Fiori",
            "Jardim Maria do Carmo",
            "Vila Gabriel",
            "Vila Progresso",
            "Vila Porto",
            "Vila Rica",
            "Parque Vitória Régia",
            "Retiro São João",
            "Jardim Iguatemi",
            "Jardim Morumbi",
            "Jardim Leocádia",
            "Vila Cardoso",
            "Vila Barão",
            "Vila Nova Esperança",
            "Jardim Zulmira",
            "Jardim Simus",
            "Jardim Vera Cruz",
            "Santa Terezinha",
            "Vila Trujillo",
            "Vila Carvalho",
            "Vila Santana",
            "Jardim Santa Rosália"
        ]
    },
    "oeste": {
        "nome": "Zona Oeste",
        "bairros": [
            "Jardim Santa Bárbara",
            "Júlio de Mesquita Filho",
            "Jardim Wanel Ville",
            "Parque Esmeralda",
            "Jardim Ouro Fino",
            "Parque Ouro Fino",
            "Parque Manchester",
            "Central Parque",
            "Largo do Divino",
            "Jardim São Paulo"
        ]
    },
    "sul": {
        "nome": "Zona Sul",
        "bairros": [
            "Vila Jardini",
            "Jardim Magnólia",
            "Jardim Paulistano",
            "Jardim Europa",
            "Jardim dos Estados",
            "Jardim América",
            "Jardim Portal da Colina",
            "Jardim Emília",
            "Jardim Sandra",
            "Parque da Biquinha",
            "Vila Assis",
            "Parque Água Vermelha",
            "Jardim Guadalajara",
            "Jardim São Carlos",
            "Parque Campolim"
        ]
    },
    "leste": {
        "nome": "Zona Leste",
        "bairros": [
            "Jardim Gonçalves",
            "Jardim Sol",
            "Jardim Bandeirantes",
            "Jardim Piratininga",
            "Jardim Granja",
            "Vila Haro",
            "Parque Três Meninos",
            "Vila Hortência",
            "Jardim Gutierres",
            "Jardim Prestes de Barros",
            "Vila Colorau",
            "Vila Barcelona",
            "Jardim Parada do Alto"
        ]
    },
    "industrial": {
        "nome": "Zona Industrial",
        "bairros": [
            "Aparecidinha",
            "Brigadeiro Tobias",
            "Alto da Boa Vista",
            "Parque Chico Mendes"
        ]
    }
}

import math, time, json, requests

# Cor fixa de cada zona (a MESMA usada na legenda -> cores sempre batem):
CORES = {"centro":"#1f77b4","norte":"#ff7f0e","oeste":"#2ca02c",
         "sul":"#d62728","leste":"#9467bd","industrial":"#8c564b"}

def dist_km(lon1, lat1, lon2, lat2):
    """Distância aproximada (equiretangular) em km — ótima em escala de cidade."""
    R = 6371.0
    x = math.radians(lon2 - lon1) * math.cos(math.radians((lat1 + lat2)/2))
    y = math.radians(lat2 - lat1)
    return R * math.hypot(x, y)

def atribuir_por_proximidade(setores_xy, pontos):
    """Cada setor (id, lon, lat) recebe a zona do bairro mais próximo."""
    out = []
    for sid, slon, slat in setores_xy:
        melhor = None
        for zona, bairro, blon, blat in pontos:
            d = dist_km(slon, slat, blon, blat)
            if melhor is None or d < melhor[0]:
                melhor = (d, zona, bairro)
        out.append({"id": sid, "zona": melhor[1], "bairro": melhor[2],
                    "dist_km": round(melhor[0], 3)})
    return out

print("Funções de recorte carregadas.")

In [ ]:
# 3) Baixa a malha de setores censitários de Sorocaba (IBGE, via geobr)
import geobr, warnings
from shapely.geometry import Point
warnings.filterwarnings("ignore")
COD_SOROCABA = 3552205
try:
    setores = geobr.read_census_tract(code_tract=COD_SOROCABA, year=2022,
                                      simplified=False, showProgress=False)
except TypeError:
    setores = geobr.read_census_tract(code_tract=COD_SOROCABA, year=2022)
setores = setores.to_crs(4326)
col_id = next((c for c in ["code_tract","CD_SETOR","CD_GEOCODI"] if c in setores.columns),
              setores.columns[0])
setores["CD_SETOR"] = setores[col_id].astype(str)

# Contorno de Sorocaba (para travar a busca de bairros dentro da cidade):
SORO_POLY = setores.geometry.union_all() if hasattr(setores.geometry, "union_all") \
            else setores.geometry.unary_union
minx, miny, maxx, maxy = SORO_POLY.bounds
print(f"Setores de Sorocaba: {len(setores):,}")
print(f"Caixa da cidade: lon [{minx:.3f}, {maxx:.3f}]  lat [{miny:.3f}, {maxy:.3f}]")

In [ ]:
# 4) Descobre a posição de cada bairro pelo OpenStreetMap — DENTRO de Sorocaba
CACHE = "/content/geocode_bairros.json"
try:
    cache = json.load(open(CACHE, encoding="utf-8"))
except Exception:
    cache = {}

headers = {"User-Agent": "CityLens-Sorocaba/1.0 (projeto escolar)"}
# viewbox = canto1(lon,lat), canto2(lon,lat) + bounded=1 -> só resultados na caixa
viewbox = f"{minx},{maxy},{maxx},{miny}"

pontos, faltas, fora = [], [], []
for zona, z in ZONAS.items():
    for bairro in z["bairros"]:
        chave = f"{bairro}|Sorocaba|SP|v2"
        if chave not in cache:
            try:
                r = requests.get("https://nominatim.openstreetmap.org/search",
                                 params={"q": f"{bairro}, Sorocaba, SP, Brasil",
                                         "format": "json", "limit": 1,
                                         "countrycodes": "br",
                                         "viewbox": viewbox, "bounded": 1},
                                 headers=headers, timeout=25)
                js = r.json()
                cache[chave] = [float(js[0]["lon"]), float(js[0]["lat"])] if js else None
            except Exception as e:
                print("Erro de rede ao geocodificar", bairro, "->", e)
                cache[chave] = None
            json.dump(cache, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)
            time.sleep(1.1)   # política do Nominatim: no máx. 1 pedido por segundo
        coord = cache[chave]
        if not coord:
            faltas.append((zona, bairro))
        elif not SORO_POLY.contains(Point(coord[0], coord[1])):
            fora.append((zona, bairro))            # caiu fora da cidade -> descarta
        else:
            pontos.append((zona, bairro, coord[0], coord[1]))

print(f"Bairros usados: {len(pontos)} | não encontrados: {len(faltas)} | fora de Sorocaba (descartados): {len(fora)}")
if faltas:
    print("  não encontrados:", ", ".join(f"{b} [{ZONAS[z]['nome']}]" for z, b in faltas))
if fora:
    print("  fora da cidade :", ", ".join(f"{b} [{ZONAS[z]['nome']}]" for z, b in fora))
print("\nPontos por zona:")
from collections import Counter
pc = Counter(p[0] for p in pontos)
for z in ZONAS: print(f"  {ZONAS[z]['nome']:<16} {pc.get(z,0):>3} bairros")
vazias = [ZONAS[z]["nome"] for z in ZONAS if pc.get(z,0)==0]
assert not vazias, f"Zona sem nenhum bairro válido: {vazias} — confira os nomes em ZONAS."

In [ ]:
# 5) Atribui cada setor à sua zona e desenha o mapa (cores = legenda)
cent = setores.to_crs(3857).geometry.centroid.to_crs(4326)
setores_xy = [(cd, p.x, p.y) for cd, p in zip(setores["CD_SETOR"], cent)]
atrib = atribuir_por_proximidade(setores_xy, pontos)

zona_de = {a["id"]: a["zona"] for a in atrib}
ordem = list(ZONAS.keys())
setores["zona"] = setores["CD_SETOR"].map(zona_de)
setores["cor"]  = setores["zona"].map(CORES).fillna("#cccccc")

import matplotlib.pyplot as plt, matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(9, 9))
setores.plot(color=setores["cor"], linewidth=0.12, edgecolor="white", ax=ax)
ax.legend(handles=[mpatches.Patch(color=CORES[z], label=ZONAS[z]["nome"]) for z in ordem],
          loc="lower left", fontsize=9, frameon=True)
ax.set_title("Sorocaba — setores atribuídos às 6 zonas (por proximidade)")
ax.set_axis_off(); plt.show()

from collections import Counter
c = Counter(a["zona"] for a in atrib)
print("Setores por zona:")
for z in ordem:
    print(f"  {ZONAS[z]['nome']:<16} {c.get(z,0):>5}")

import csv
with open("/content/setores_por_zona.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["CD_SETOR","zona","bairro_mais_proximo","dist_km"])
    for a in atrib: w.writerow([a["id"], a["zona"], a["bairro"], a["dist_km"]])
print("\nCSV salvo em /content/setores_por_zona.csv")